In [1]:
import gzip
import shutil
import time

import pandas as pd
import requests
import torch
import torch.nn.functional as F
import torch.nn as nn

url = "https://github.com/rasbt/machine-learning-book/raw/main/ch08/movie_data.csv.gz"
filename = url.split("/")[-1]

with open(filename, "wb") as f:
    r = requests.get(url)
    f.write(r.content)

with gzip.open('movie_data.csv.gz', 'rb') as f_in:
    with open('movie_data.csv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

In [2]:
df = pd.read_csv('movie_data.csv')
df.head()

In [ ]:
train_texts = df.iloc[:35000]['review'].values
train_labels = df.iloc[:35000]['sentiment'].values

valid_texts = df.iloc[35000:40000]['review'].values
valid_labels = df.iloc[35000:40000]['sentiment'].values

test_texts = df.iloc[40000:]['review'].values
test_labels = df.iloc[40000:]['sentiment'].values

In [3]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
valid_encodings = tokenizer(list(valid_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [4]:
class MHAPyTorchScaledDotProduct(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out is indivisible by num_heads"

        self.num_heads = num_heads
        self.context_length = context_length
        self.head_dim = d_out // num_heads
        self.d_out = d_out

        self.qkv = nn.Linear(d_in, 3 * d_out, bias=qkv_bias)
        self.proj = nn.Linear(d_out, d_out)
        self.dropout = dropout

    def forward(self, x, attention_mask=None):
        batch_size, num_tokens, embed_dim = x.shape

        # (b, num_tokens, embed_dim) --> (b, num_tokens, 3 * embed_dim)
        qkv = self.qkv(x)

        # (b, num_tokens, 3 * embed_dim) --> (b, num_tokens, 3, num_heads, head_dim)
        qkv = qkv.view(batch_size, num_tokens, 3, self.num_heads, self.head_dim)

        # (b, num_tokens, 3, num_heads, head_dim) --> (3, b, num_heads, num_tokens, head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # (3, b, num_heads, num_tokens, head_dim) -> 3 times (b, num_heads, num_tokens, head_dim)
        queries, keys, values = qkv

        # Attention mask'i doğru formata çevir
        attn_mask = None

        if attention_mask is not None:
            # (b, num_tokens) -> (b, 1, 1, num_tokens) -> (b, num_heads, num_tokens, num_tokens)
            attn_mask = attention_mask.unsqueeze(1).unsqueeze(2)
            attn_mask = attn_mask.expand(batch_size, self.num_heads, num_tokens, num_tokens)
            attn_mask = attn_mask.bool()

        use_dropout = 0. if not self.training else self.dropout

        context_vec = nn.functional.scaled_dot_product_attention(
            queries, keys, values, attn_mask=attn_mask, dropout_p=use_dropout, is_causal=False)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.d_out)

        context_vec = self.proj(context_vec)

        return context_vec


In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.2):
        super().__init__()
        self.attn = MHAPyTorchScaledDotProduct(embed_dim, embed_dim, num_heads, context_length=128, dropout=dropout)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 4*embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(4*embed_dim, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        x = x + self.dropout(self.attn(self.ln1(x), attention_mask))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

In [19]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=8, num_layers=8, num_classes=2, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embeddings = nn.Embedding(max_len, embed_dim)
        self.layers = nn.ModuleList([TransformerBlock(embed_dim, num_heads) for _ in range(num_layers)])
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids, attention_mask=None):
        b, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(b, seq_len)
        x = self.embedding(input_ids) + self.position_embeddings(positions)

        for layer in self.layers:
            x = layer(x, attention_mask)

        x = x.transpose(1,2)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)

In [7]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [8]:
train_dataset = IMDbDataset(train_encodings, train_labels)
valid_dataset = IMDbDataset(valid_encodings, valid_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=16, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False)

In [9]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = tokenizer.vocab_size
model = SentimentClassifier(vocab_size).to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

In [20]:
from tqdm import tqdm

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        if (i + 1) % 500 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    # Epoch sonunda validation
    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, valid_loader, device)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Valid Loss: {val_loss:.4f}, Valid Accuracy: {val_acc:.4f}\n")

# Final test evaluation
test_loss, test_acc = evaluate(model, test_loader, device)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Epoch 1/3:  23%|██▎       | 510/2188 [00:05<00:18, 90.08it/s]

Epoch [1/3], Step [500/2188], Loss: 0.5998


Epoch 1/3:  46%|████▋     | 1015/2188 [00:11<00:13, 89.36it/s]

Epoch [1/3], Step [1000/2188], Loss: 0.3632


Epoch 1/3:  69%|██████▉   | 1513/2188 [00:16<00:07, 89.73it/s]

...

Epoch [3/3], Step [1500/2188], Loss: 0.1678


Epoch 3/3:  92%|█████████▏| 2017/2188 [00:22<00:01, 89.62it/s]

Epoch [3/3], Step [2000/2188], Loss: 0.2927


Epoch 3/3: 100%|██████████| 2188/2188 [00:24<00:00, 89.50it/s]



Epoch [3/3]
Train Loss: 0.2763
Valid Loss: 0.3560, Valid Accuracy: 0.8516

Test Loss: 0.3680, Test Accuracy: 0.8414


In [21]:
def print_predictions(model, texts, labels, tokenizer, device, num_samples=5):
    model.eval()

    for i in range(min(num_samples, len(texts))):
        text = texts[i]
        true_label = labels[i]


        encoding = tokenizer(text, truncation=True, padding=True, return_tensors='pt')
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)


        with torch.no_grad():
            outputs = model(input_ids, attention_mask)
            probs = F.softmax(outputs, dim=1)
            predicted = torch.argmax(outputs, dim=1).item()
            confidence = probs[0][predicted].item()

        sentiment_map = {0: "Negative", 1: "Positive"}

        print(f"\n{'='*80}")
        print(f"Example {i+1}:")
        print(f"Text: {text[:200]}..." if len(text) > 200 else f"Text: {text}")
        print(f"\nTrue Label: {sentiment_map[true_label]}")
        print(f"Predicted: {sentiment_map[predicted]} (Confidence: {confidence:.2%})")
        print(f"Correct: {'✓' if predicted == true_label else '✗'}")

print("\n" + "="*80)
print("SAMPLE PREDICTIONS FROM TEST SET")
print("="*80)
print_predictions(model, test_texts, test_labels, tokenizer, device, num_samples=10)


SAMPLE PREDICTIONS FROM TEST SET

Example 1:
Text: Such energy and vitality. You just can't go wrong with Busby Berkley films and this certainly must be his best. Of course the choreography is wonderful, but also the banter between Cagney and Blondell...

True Label: Positive
Predicted: Positive (Confidence: 98.83%)
Correct: ✓

Example 2:
Text: but Thomas Ian Griffith just doesn't have the polish that a big bucks actor has, granted this was made 5+ years ago. Some of the humorous lines could have been timed to make this not only action, but ...

True Label: Negative
Predicted: Negative (Confidence: 99.95%)
Correct: ✓

Example 3:
Text: Why this film was only released in 4 states is beyond me. I thought this film was a divine story. The name says it all: Seeing Other People. This movie has more logic than laughs, which I suppose is w...

True Label: Positive
Predicted: Positive (Confidence: 92.52%)
Correct: ✓

Example 4:
Text: I'm not here to tell you "Armored" is Kubrickian, Hitchcocki